# JP-1：組から終わった行を抜く確認（案A・GPU・Sarashina2.2 1B・float32）

日本語A では、32本の組の約4割で1本だけが上限まで走り、組全体がそれに付き合っていました。案A では、**終わった行を組から抜いて**、残った行だけを計算します。引き方・乱数・上限・q の定義は変えません。このノートブックはその確認用で、計画のゲートではありません。

- 日本語A の本番の4か所（上限まで走った組を3つ、EOS で終わった組を1つ。決まった規則で選びます）で、**本番と同じやり方**と**抜くやり方**をそれぞれ回し、時間を比べます。
- どちらも、本番で保存した結果（32本のハッシュ・答え・q・監査の4本）と比べます。
- 抜くやり方を2回回して、毎回同じになるかも見ます。
- 頼み方を作るために MGSM を読み込みますが、画面とファイルに出すのは数・真偽・ハッシュだけです。

**使い方**：ランタイムは **GPU（L4）**。シークレットは `GITHUB_TOKEN`。「すべてのセルを実行」で **20〜30 分くらい**。結果は Drive の `DECISION_POINTS/jp1/SHRINKCHECK/`。終わったら Claude に教えてね。

- **通知**：スマホの ntfy アプリで `godpork-b06080cfb3cdfa8f` を購読しておくと、始まったとき・終わったとき・エラーで止まったときに届きます。ランタイムごと切れたときは届かないので、そのときは Colab を見てください。
- **終わったら自動でランタイムを切ります**（CU の節約）。Drive への保存を確定してから切るので、結果は残ります。エラーで止まったときは切らずに残します。切りたくないときは、最初に `AUTO_DISCONNECT = False` を実行してください。
- **エラーのとき**：通知の本文にエラーの文面（トレースバック）が入ります。全文は Drive の `DECISION_POINTS/_errors/` にも保存されるので、Claude が直接読めます。

In [ ]:
# Cell 1 — Drive, GPU, exact head, notifications.
from google.colab import drive, userdata
from pathlib import Path
import os, shutil, stat, subprocess, sys, json, time, hashlib, datetime as dt

# 通知（ntfy）：スマホの ntfy アプリでこの名前を購読しておくと、始まったとき・終わったとき・止まったときに届きます。エラーのときはエラーの文面も送ります（鍵・トークンらしい文字は伏せ字。全文は Drive の DECISION_POINTS/_errors）
import urllib.request
NTFY_TOPIC = 'godpork-b06080cfb3cdfa8f'
NB_LABEL = 'JP-1 組から終わった行を抜く確認（案A・GPU・Sarashina 1B・float32・日本語Aの4か所）'   # どのノートブックからの通知かの名札
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=('[' + NB_LABEL + '] ' + msg).encode('utf-8'),
                                     headers={'Title': 'Pork - JP-1 shrink check', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
import os, re as _re, traceback as _tb, datetime as _edt
ERR_SLUG = 'jp1_shrinkcheck'   # Drive のエラーファイルの名前の頭
_SECRET_RE = _re.compile(r"sk-ant-[A-Za-z0-9_\-]+|github_pat_[A-Za-z0-9_]+|gh[pousr]_[A-Za-z0-9]{8,}|hf_[A-Za-z0-9]{8,}|x-access-token:[^@\s]+@")
def _scrub(s):
    s = _SECRET_RE.sub('[伏せ字]', s)
    return '\n'.join(l if len(l) <= 300 else l[:300] + ' …（長いので省略）' for l in s.splitlines())
def _error_text(result):
    e = result.error_in_exec if result.error_in_exec is not None else result.error_before_exec
    tb = ''.join(_tb.format_exception(type(e), e, e.__traceback__))
    first = (getattr(getattr(result, 'info', None), 'raw_cell', '') or '').strip().splitlines()[:1]
    return _scrub('セル: ' + (first[0][:100] if first else '?') + '\n' + tb)
def _save_error(text):
    base = '/content/drive/MyDrive'
    if not os.path.isdir(base):
        return None
    d = os.path.join(base, 'DECISION_POINTS', '_errors')
    os.makedirs(d, exist_ok=True)
    stem = ERR_SLUG + '_' + _edt.datetime.now(_edt.timezone(_edt.timedelta(hours=9))).strftime('%Y%m%d_%H%M%S')
    name, n = stem + '.txt', 1
    while os.path.exists(os.path.join(d, name)):
        n += 1
        name = stem + '_' + str(n) + '.txt'
    with open(os.path.join(d, name), 'w', encoding='utf-8') as fh:
        fh.write('[' + NB_LABEL + ']\n' + text)
    return 'DECISION_POINTS/_errors/' + name
def _short(text, limit=3300):
    b = text.encode('utf-8')
    if len(b) <= limit:
        return text
    return '…（前のほうは省略。全文は Drive）\n' + b[-limit:].decode('utf-8', 'ignore')
def _on_error(result):
    if result.error_in_exec is None and result.error_before_exec is None:
        return
    try:
        text = _error_text(result)
    except Exception as e2:
        text = 'エラーの文面を作れませんでした：' + type(e2).__name__
    try:
        where = _save_error(text)
    except Exception as e3:
        where = None
        text += '\n（Drive に保存できませんでした：' + type(e3).__name__ + '）'
    print('エラーの全文を通知に送ります' + ('（Drive：' + where + '）' if where else ''))
    notify('⚠ 止まりました：エラーが出ています。' + ('全文は Drive の ' + where if where else 'Drive には保存できませんでした')
           + '\n\n' + _short(text), tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('🚀 スタートしました（これが届けば通知の設定は OK です）', tags='rocket')

os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

drive.mount("/content/drive")
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
assert gpu.returncode == 0 and gpu.stdout.strip(), "GPU のランタイム（L4）にしてね"
GPU_NAME = gpu.stdout.strip().split(",")[0]
EXPECTED_HEAD = "95563109abdb1817a660c60105c61ff3e79c1646"
REPO_URL = "https://github.com/chichichi2001/Pork.git"
REPO = Path("/content/Pork")

def get_secret(name):
    try:
        v = userdata.get(name)
    except Exception:
        v = None
    return str(v).strip() if v and str(v).strip() else None

token = get_secret("GITHUB_TOKEN")
assert token, "Colab Secrets に GITHUB_TOKEN を登録してください。"
askpass = Path("/tmp/pork_git_askpass.sh")
askpass.write_text('#!/bin/sh\ncase "$1" in\n  *Username*) echo "x-access-token" ;;\n  *Password*) echo "$GITHUB_TOKEN" ;;\n  *) echo "" ;;\nesac\n')
askpass.chmod(askpass.stat().st_mode | stat.S_IXUSR)
env = dict(os.environ, GITHUB_TOKEN=token, GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT="0")
os.chdir('/content')   # セル1 をやり直しても、消すフォルダの中にいないように（2026-10-14）
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--branch", "decision-points", "--single-branch", REPO_URL, str(REPO)], check=True, env=env)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", EXPECTED_HEAD], check=True)
head = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert head == EXPECTED_HEAD, (head, EXPECTED_HEAD)
token = None
env.pop("GITHUB_TOKEN", None)
os.chdir(REPO)
sys.path.insert(0, str(REPO))
THEME = Path("/content/drive/MyDrive/DECISION_POINTS")
SRC = THEME / "jp1" / "MAIN" / "JP1_ja_A"      # read only
D = THEME / "jp1" / "SHRINKCHECK"
MODEL_DIR = Path("/content/sarashina_1b")
D.mkdir(parents=True, exist_ok=True)
JST = dt.timezone(dt.timedelta(hours=9))
STARTED = time.time()
print("GPU:", GPU_NAME, "| head:", head)

In [ ]:
# Cell 2 — versions, precision, ja items (hash as G0 run 2), ja-A units (validated), the 4 points by a fixed rule.
from pork import jp1 as P
PINS = [f"{k}=={v}" for k, v in P.PACKAGES.items()] + ["protobuf"]
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], capture_output=True, text=True)
assert r.returncode == 0, r.stderr[-3000:]
import importlib.metadata as md
import torch as torch_mod
from pork import jp1_runner as rr
from pork.quality_pork_q3a_runner import atomic_json
CHECKS = {"packages": {k: md.version(k) for k in P.PACKAGES}, "torch": md.version("torch"), "gpu": GPU_NAME,
          "precision_flags": P.precision_flags(torch_mod), "gate_doc_sha256_ok": P.gate_doc_sha256(REPO) == P.GATE_DOC_SHA256}
ID = json.loads((SRC / "RUN_IDENTITY.json").read_text(encoding="utf-8"))
G0C = json.loads((THEME / "jp1" / "G0_RUN2" / "G0_CHECKS.json").read_text(encoding="utf-8"))["checks"]
ITEMS = P.items(P.load_rows("ja"), "main")
ITEMS_SHA = P.items_sha256(ITEMS)
CHECKS["items_ok"] = ITEMS_SHA == G0C["items_sha256"]["ja"] == ID["items_sha256"]
CHECKS["same_gpu_as_run"] = ID["gpu_name"] == GPU_NAME
VERSION_OK = (CHECKS["packages"] == P.PACKAGES and CHECKS["torch"].split("+")[0] == P.TORCH_BASE_VERSION
              and P.precision_ok(CHECKS["precision_flags"]) and CHECKS["gate_doc_sha256_ok"] and CHECKS["items_ok"]
              and CHECKS["same_gpu_as_run"])
print(json.dumps(CHECKS, indent=1))
assert VERSION_OK, P.STOP_VERSION + " — 推論の前に止めました。Claude に見せてね。"
RD = lambda kind, row, i: rr.read_envelope(SRC / "units" / (f"base_{row:03d}.json" if kind == "base" else f"point_{row:03d}_{i:03d}.json"),
                                           "ja", ID["model"], kind, row, i, ID["code_commit"], ID["items_sha256"], tries=1)
BASES = {row: RD("base", row, None) for row in P.HALVES["A"]}
cand_budget, cand_eos = [], []
for row in P.HALVES["A"]:
    for i in range(len(P.points(rr.answer_pos(BASES[row])))):
        keep = RD("point", row, i)["kinds"]["keep"]
        (cand_eos if all(keep["eos"]) else cand_budget).append((row, i))
pick = lambda c, n: [c[round(k * (len(c) - 1) / (n - 1))] for k in range(n)] if n > 1 else [c[len(c) // 2]]
POINTS = pick(cand_budget, 3) + pick(cand_eos, 1)
print("candidates (keep ran to budget / ended by EOS):", len(cand_budget), len(cand_eos), "| picked:", POINTS)

In [ ]:
# Cell 3 — the main run's step vs the shrinking batch on the 4 points; both compared with the stored units; timing.
from pork import kv_cached_backend as kv
from pork import pork_t_v1 as pt
from pork import jp1_shared_prefix as SP
from pork import jp1_shrinking_batch as SB
from huggingface_hub import snapshot_download
SPEC = P.MODELS[ID["model"]]
snapshot_download(SPEC["id"], revision=SPEC["revision"], local_dir=str(MODEL_DIR), token=False,
                  allow_patterns=["*.json", "*.safetensors", "tokenizer*", "*.model"])
_t, TOKC, MODEL = rr.load_model(str(MODEL_DIR), None, "cuda", SPEC)
assert P.model_ok(MODEL, SPEC) and P.precision_ok(P.precision_flags(torch_mod))
EOS = kv.eos_ids(TOKC, MODEL)
assert list(EOS) == ID["eos_ids"], "EOS が本番と違います — Claude に見せてね"
Bt = pt.backend("torch", "cuda")
chat_ids = lambda text: kv.chat_prompt_ids(TOKC, text)
decode = lambda ids: TOKC.decode(list(ids), skip_special_tokens=True)
OLD_FACTORY = SP.torch_shared_prefix_batch(MODEL, torch_mod, "cuda")     # the main run's step
FNS = SB.torch_fns(MODEL, torch_mod, "cuda")
_orig_gb = rr.generate_batch
def _shrinking_generate_batch(B, batch_logits_fn, prefix_ids, streams, eos, temperature, max_new, draw=None):
    return SB.generate_shrinking(B, prefix_ids=prefix_ids, streams=streams, eos=eos, temperature=temperature,
                                 max_new=max_new, draw=draw, **FNS)
def run_point(row, i, shrinking):
    rr.generate_batch = _shrinking_generate_batch if shrinking else _orig_gb
    try:
        torch_mod.cuda.synchronize(); t0 = time.time()
        res = rr.compute_point("ja", row, i, BASES[row], ITEMS[row], B=Bt, chat_ids=chat_ids, batch_fn_factory=OLD_FACTORY,
                               decode=decode, eos=EOS)
        torch_mod.cuda.synchronize()
        return res, time.time() - t0
    finally:
        rr.generate_batch = _orig_gb
def compare(new, stored):
    out = {}
    for k, s in stored["kinds"].items():
        n = new["kinds"][k]
        out[k] = {"all32_sha_same": n["token_ids_sha256"] == s["token_ids_sha256"],
                  "predictions_same": sum(a == b for a, b in zip(n["predictions"], s["predictions"])),
                  "eos_same": sum(a == b for a, b in zip(n["eos"], s["eos"])),
                  "audit_rows_same": sum(a == b for a, b in zip(n["audit_token_ids"], s["audit_token_ids"])),
                  "q_same": n["q"] == s["q"], "q_abs_diff": abs(n["q"] - s["q"]) if None not in (n["q"], s["q"]) else None}
    return out
ROWS = []
for row, i in POINTS:
    stored = RD("point", row, i)
    old, t_old = run_point(row, i, False)
    new, t_new = run_point(row, i, True)
    ROWS.append({"row": row, "i": i, "t": stored["t"], "kinds": sorted(stored["kinds"]), "seconds_old": t_old,
                 "seconds_new": t_new, "old_vs_stored": compare(old, stored), "new_vs_stored": compare(new, stored),
                 "new_vs_old": compare(new, old)})
    print(json.dumps({"row": row, "i": i, "seconds_old": round(t_old, 1), "seconds_new": round(t_new, 1),
                      "old_vs_stored_all32": {k: v["all32_sha_same"] for k, v in ROWS[-1]["old_vs_stored"].items()},
                      "new_vs_old_all32": {k: v["all32_sha_same"] for k, v in ROWS[-1]["new_vs_old"].items()},
                      "new_vs_old_predictions_same": {k: v["predictions_same"] for k, v in ROWS[-1]["new_vs_old"].items()}}))
# run-to-run identity of the shrinking batch on the first point
r0, i0 = POINTS[0]
a, _ = run_point(r0, i0, True)
b, _ = run_point(r0, i0, True)
REPEAT = {"row": r0, "i": i0, "identical": all(a["kinds"][k]["token_ids_sha256"] == b["kinds"][k]["token_ids_sha256"] for k in a["kinds"])}
T_OLD, T_NEW = sum(x["seconds_old"] for x in ROWS), sum(x["seconds_new"] for x in ROWS)
RES = {"schema": "jp1_shrink_check_v1", "head": EXPECTED_HEAD, "gpu": GPU_NAME, "checks": CHECKS, "points": ROWS,
       "repeat": REPEAT, "seconds_old_total": T_OLD, "seconds_new_total": T_NEW, "new_over_old": T_NEW / T_OLD,
       "source": "jp1/MAIN/JP1_ja_A", "run_code_commit": ID["code_commit"],
       "finished_jst": dt.datetime.now(JST).isoformat(timespec="seconds"), "elapsed_hours": (time.time() - STARTED) / 3600}
atomic_json(D / "SHRINK_SUMMARY.json", RES)
print(json.dumps({"seconds_old_total": round(T_OLD), "seconds_new_total": round(T_NEW), "new_over_old": round(T_NEW / T_OLD, 3),
                  "repeat_identical": REPEAT["identical"]}, indent=1))
notify("🎉 組から抜く確認 終わりました（結果は Drive の jp1/SHRINKCHECK）", tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    print('Drive への書き込みを確定してから、ランタイムを切ります（CU の節約。止めたいときは AUTO_DISCONNECT = False）')
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount()
    time.sleep(3)
    _runtime.unassign()